# OpenAI baseline via OpenRouter (gpt-4o-mini)

1. Put your OpenRouter key in the project `.env` file:
   `OPENAI_API_KEY=sk-or-v1-...` (or `OPENROUTER_API_KEY=...`)
2. In the config cell: set `DATA_VERSION`, `DATA_VARIANT`, and `MAX_SAMPLES`

Inputs: files live in the same directory as this notebook.

Outputs: `openai_translation/`

In [1]:
%pip install -q openai sacrebleu tqdm python-dotenv
print("Restart kernel, then run from the config cell.")

Note: you may need to restart the kernel to use updated packages.
Restart kernel, then run from the config cell.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vnpnk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path

ROOT = Path.cwd()

# --- user settings ---

# "v1" or "v2"
DATA_VERSION = "v1"

# None -> ende_dev_v1.jsonl; "expand" -> ende_dev_v1_expand.jsonl; "cleaned" -> ende_dev_v1_cleaned.jsonl
DATA_VARIANT = None

# None = all; e.g. 10 for a quick smoke test
MAX_SAMPLES = 2

LANG_GROUPS = ["ende", "enru", "enes"]
MODES = ["no_term", "proper_term", "random_term"]

LANG_CONFIG = {
    "ende": {"ref_field": "de", "target_lang": "German", "output_tag": "de"},
    "enru": {"ref_field": "ru", "target_lang": "Russian", "output_tag": "ru"},
    "enes": {"ref_field": "es", "target_lang": "Spanish", "output_tag": "es"},
}

OUTPUT_BASE = (ROOT / "openai_translation").resolve()

# Few-shot examples from *_dev_v2.jsonl.
SAMPLE_SENTENCES: dict[str, list[dict[str, object]]] = {
    "ende": [
        {"en": "The status of each individual space can be seen from the color code on the upper left corner.", "de": "Der Farbcode oben links gibt den Status des betreffenden Space an.", "proper_terms": {"space": "Space"}, "random_terms": {"status": "Status"}},
        {"en": "This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.", "de": "Dieser Service beschreibt den implementierten Zustand (Laufzeitzustand) von SAP-HANA-Datenbankartefakten, z. B. Tabellen, Views oder Prozeduren, die von den SAP-Integrated-Development-Environment-Editoren (WebIDE-Editoren) als eine Familie konsistenter Entwurfszeit-Artefakte für alle wichtigen SAP-HANA-Plattform-Datenbankfunktionen erstellt oder angepasst wurden.", "proper_terms": {"design": "Entwurf", "state": "Zustand"}, "random_terms": {"service": "Service", "features": "funktionen"}},
        {"en": "Your contract includes a total number of available capacity units per month which you can allocate as you wish to the compute and storage resources.", "de": "Ihr Vertrag enthält eine Gesamtzahl verfügbarer Kapazitätseinheiten pro Monat, die Sie den Rechen- und Speicherressourcen nach Belieben zuweisen können.", "proper_terms": {"storage": "Speicher"}, "random_terms": {"total": "Gesamtzahl", "available": "verfügbarer"}}
    ],
    "enru": [
        {"en": "To add notes, choose a template and open the Notes pane on the left side of the screen.", "ru": "Чтобы добавить примечания, выберите шаблон и откройте область Примечания на левой стороне экрана.", "proper_terms": {"pane": "область"}, "random_terms": {"add": "добавить", "choose": "выберите"}},
        {"en": "Indicates if a configuration item or configuration step is specific to a localized solution version.", "ru": "Указывает, являются ли позиция или шаг конфигурации специфичными для локализованной версии решения.", "proper_terms": {"item": "позиция"}, "random_terms": {"version": "версии", "solution": "решения"}},
        {"en": "Specifies the number of the contract from which you can select service items.", "ru": "Указывает номер контракта, из которого можно выбрать позиции услуг.", "proper_terms": {"contract": "контракт"}, "random_terms": {"number": "номер", "select": "выбрать"}}
    ],
    "enes": [
        {"en": "Why would you need to access HDI containers?", "es": "¿Por qué tendría que acceder a los containers HDI?", "proper_terms": {"container": "container"}, "random_terms": {"access": "acceder"}},
        {"en": "In such cases you may use the Move Items or Merge feature.", "es": "En estos casos, puede utilizar la función Mover elementos o Fusionar .", "proper_terms": {"item": "elemento"}, "random_terms": {"may": "puede"}},
        {"en": "Decide if you want to use parallel processing for this job:", "es": "Decida si desea utilizar el procesamiento paralelo para este job:", "proper_terms": {"processing": "procesamiento", "job": "job", "parallel processing": "procesamiento paralelo"}, "random_terms": {"Decide": "Decida", "want": "desea"}}
    ],
}


def data_stem(lang: str) -> str:
    stem = f"{lang}_dev_{DATA_VERSION}"
    if DATA_VARIANT is not None:
        stem = f"{stem}_{DATA_VARIANT}"
    return stem


def data_path(lang: str) -> Path:
    return ROOT / f"{data_stem(lang)}.jsonl"


def prediction_stem(lang: str) -> str:
    return data_stem(lang)


print("DATA_VERSION:", DATA_VERSION)
print("DATA_VARIANT:", DATA_VARIANT)
print("DATA_DIR:", ROOT)
print("MAX_SAMPLES:", "all" if MAX_SAMPLES is None else MAX_SAMPLES)
print("OUTPUT_BASE:", OUTPUT_BASE)
for lang in LANG_GROUPS:
    path = data_path(lang)
    status = "ok" if path.exists() else "MISSING"
    print(f"  {lang}: {path.name} [{status}] ({len(SAMPLE_SENTENCES[lang])} prompt examples)")

DATA_VERSION: v1
DATA_VARIANT: None
DATA_DIR: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline
MAX_SAMPLES: 2
OUTPUT_BASE: C:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openai_translation
  ende: ende_dev_v1.jsonl [ok] (3 prompt examples)
  enru: enru_dev_v1.jsonl [ok] (3 prompt examples)
  enes: enes_dev_v1.jsonl [ok] (3 prompt examples)


In [6]:
import os
import time

from dotenv import load_dotenv
from openai import OpenAI

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


def find_env_file(start: Path) -> Path | None:
    for directory in [start, *start.parents]:
        env_path = directory / ".env"
        if env_path.exists():
            return env_path
    return None


env_file = find_env_file(ROOT)
if env_file:
    load_dotenv(env_file)
    print("Loaded env from:", env_file)
else:
    print("No .env file found — using environment variables only")

API_KEY = (
    os.environ.get("OPENROUTER_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
    or ""
).strip()
if not API_KEY:
    raise EnvironmentError(
        "No API key found. Set OPENROUTER_API_KEY or OPENAI_API_KEY in .env or your environment."
    )

MODEL_NAME = os.environ.get("OPENROUTER_MODEL", "openai/gpt-4o-mini")
MAX_RETRIES = 3
RETRY_DELAY_S = 5

client = OpenAI(api_key=API_KEY, base_url=OPENROUTER_BASE_URL)
print("Provider: OpenRouter")
print("Base URL:", OPENROUTER_BASE_URL)
print("Model:", MODEL_NAME)
print("API key: set (not shown)")

Loaded env from: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\.env
Provider: OpenRouter
Base URL: https://openrouter.ai/api/v1
Model: openai/gpt-4o-mini
API key: set (not shown)


In [7]:
import json
import re
from collections import Counter, defaultdict
from typing import Any

import sacrebleu
from tqdm import tqdm


# --- I/O ---

def load_jsonl(path: Path, max_samples: int | None = None) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if max_samples is not None and len(records) >= max_samples:
                break
    return records


def save_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_datasets() -> dict[str, list[dict[str, Any]]]:
    datasets = {}
    for lang in LANG_GROUPS:
        path = data_path(lang)
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset for {lang}: {path}")
        datasets[lang] = load_jsonl(path, MAX_SAMPLES)
    return datasets


# --- terminology helpers ---

def terms_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str]:
    if mode == "proper_term":
        return (sample.get("proper_terms") or {}).copy()
    if mode == "random_term":
        terms = (sample.get("random_terms") or {}).copy()
        for key in (sample.get("proper_terms") or {}):
            terms.pop(key, None)
        return terms
    return {}


def terminology_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str] | None:
    terms = terms_for_mode(sample, mode)
    return terms or None


def strip_output_tags(text: str, output_tag: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(rf"</?{re.escape(output_tag)}>", "", text, flags=re.IGNORECASE).strip()


# --- metrics ---

def compute_bleu_chrf(hyps: list[str], refs: list[str]) -> dict[str, float]:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {"bleu": bleu.score, "chrf": chrf.score}


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    return len(re.findall(r"\b" + re.escape(term_norm) + r"\b", text_norm))


def terminology_accuracy(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_ratios: dict[str, float] = {}
    total_terms = 0

    for pred, sample in zip(preds, samples):
        source_text = sample.get("en", "")
        for src, tgt in terms_for_mode(sample, mode).items():
            total_terms += 1
            src_count = max(_count_term_occurrences(source_text, src), 1)
            tgt_count = _count_term_occurrences(pred, tgt)
            term_ratios[src] = min(tgt_count / src_count, 1.0)

    avg_ratio = sum(term_ratios.values()) / len(term_ratios) * 100 if term_ratios else None
    return {"total_terms": total_terms, "avg_ratio_pct": avg_ratio, "per_term_ratios": term_ratios}


def terminology_consistency(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_to_candidates: dict[str, list[str]] = defaultdict(list)

    for pred, sample in zip(preds, samples):
        for src, tgt in terms_for_mode(sample, mode).items():
            candidate = tgt if str(tgt).lower() in str(pred).lower() else "<MISSING>"
            term_to_candidates[src].append(candidate)

    per_term = {}
    macro_scores = []
    weighted_scores = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = Counter(candidates).most_common(1)[0][0]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates)
        per_term[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }
        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    return {
        "per_term": per_term,
        "macro_avg_consistency": sum(macro_scores) / len(macro_scores) if macro_scores else None,
        "weighted_avg_consistency": sum(weighted_scores) / len(weighted_scores) if weighted_scores else None,
    }


def fmt_metric(value: float | None, digits: int = 2) -> str:
    return "N/A" if value is None else f"{value:.{digits}f}"


# --- translation ---

def format_terminology_block(terms: dict[str, str]) -> str:
    if not terms:
        return ""
    return "Terminology:\n" + "\n".join(f"{s} -> {t}" for s, t in terms.items()) + "\n"


def format_sample_examples(lang: str, mode: str) -> str:
    config = LANG_CONFIG[lang]
    ref_field = config["ref_field"]
    output_tag = config["output_tag"]
    blocks = []
    for i, example in enumerate(SAMPLE_SENTENCES[lang], 1):
        term_block = format_terminology_block(terms_for_mode(example, mode))
        ref = example.get(ref_field, "")
        blocks.append(
            f"Example {i}:\n"
            f"{term_block}"
            f"Input:\n<en> {example['en']} </en>\n"
            f"Output:\n<{output_tag}> {ref} </{output_tag}>"
        )
    return "Examples:\n\n" + "\n".join(blocks) + "\n\n"


def chat_completion_with_retry(messages: list[dict[str, str]]) -> str:
    last_error: Exception | None = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY_S)
    raise RuntimeError(f"OpenRouter API failed after {MAX_RETRIES} attempts") from last_error


def translate_sample(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
) -> str:
    examples_block = format_sample_examples(lang, mode)
    term_block = format_terminology_block(terminology or {})
    if term_block:
        term_block += "\n"

    prompt = f"""You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to {target_lang}.

{examples_block}{term_block}Input:
<en> {sample_en} </en>
"""

    messages = [
        {"role": "system", "content": "You are a helpful translation assistant."},
        {"role": "user", "content": prompt},
    ]
    return chat_completion_with_retry(messages)

In [8]:
def prediction_filename(lang: str, mode: str) -> str:
    return f"{prediction_stem(lang)}_{mode}_predictions.jsonl"


def run_mode(
    lang: str,
    mode: str,
    samples: list[dict[str, Any]],
    output_dir: Path,
    config: dict[str, str],
) -> dict[str, Any]:
    ref_field = config["ref_field"]
    preds = []
    records = []

    for sample in tqdm(samples, desc=f"{lang}/{mode}"):
        pred = translate_sample(
            sample.get("en", ""),
            terminology_for_mode(sample, mode),
            config["target_lang"],
            config["output_tag"],
            lang,
            mode,
        )
        preds.append(pred)
        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred, config["output_tag"])
        records.append(record)

    clean_preds = [strip_output_tags(p, config["output_tag"]) for p in preds]
    pred_path = output_dir / prediction_filename(lang, mode)
    save_jsonl(pred_path, records)

    metrics: dict[str, Any] = {}
    if samples and ref_field in samples[0]:
        refs = [sample.get(ref_field, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))
        term_acc = terminology_accuracy(clean_preds, samples, mode)
        term_cons = terminology_consistency(clean_preds, samples, mode)
        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons

        print(
            f"[{lang}/{mode}] BLEU={fmt_metric(metrics['bleu'])} "
            f"chrF={fmt_metric(metrics['chrf'])} "
            f"term_acc={fmt_metric(term_acc['avg_ratio_pct'])}% "
            f"macro_cons={fmt_metric(term_cons['macro_avg_consistency'])} "
            f"weighted_cons={fmt_metric(term_cons['weighted_avg_consistency'])}"
        )
    else:
        print(f"[{lang}/{mode}] no reference field '{ref_field}' — metrics skipped")

    return {"predictions_file": str(pred_path), "metrics": metrics}


datasets = load_datasets()
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

summary = {
    "data_version": DATA_VERSION,
    "data_variant": DATA_VARIANT,
    "data_dir": str(ROOT),
    "prompt_examples_per_lang": {lang: len(SAMPLE_SENTENCES[lang]) for lang in LANG_GROUPS},
    "provider": "openrouter",
    "base_url": OPENROUTER_BASE_URL,
    "model": MODEL_NAME,
    "max_samples": MAX_SAMPLES,
    "languages": {},
}

for lang in LANG_GROUPS:
    config = LANG_CONFIG[lang]
    samples = datasets[lang]
    lang_dir = OUTPUT_BASE / lang
    lang_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {lang}: {len(samples)} samples → {config['target_lang']} ===")

    lang_results = {mode: run_mode(lang, mode, samples, lang_dir, config) for mode in MODES}
    summary["languages"][lang] = {
        "data_file": str(data_path(lang)),
        "sample_count": len(samples),
        **config,
        "modes": lang_results,
    }

metrics_path = OUTPUT_BASE / "metrics_summary.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone.", metrics_path)


=== ende: 2 samples → German ===


ende/no_term:   0%|          | 0/2 [00:00<?, ?it/s]

ende/no_term: 100%|██████████| 2/2 [00:02<00:00,  1.40s/it]


[ende/no_term] BLEU=87.21 chrF=93.25 term_acc=N/A% macro_cons=N/A weighted_cons=N/A


ende/proper_term: 100%|██████████| 2/2 [00:04<00:00,  2.43s/it]


...

[enes/no_term] BLEU=94.51 chrF=94.87 term_acc=N/A% macro_cons=N/A weighted_cons=N/A


enes/proper_term: 100%|██████████| 2/2 [00:01<00:00,  1.01it/s]


[enes/proper_term] BLEU=94.51 chrF=94.87 term_acc=50.00% macro_cons=1.00 weighted_cons=1.00


enes/random_term: 100%|██████████| 2/2 [00:01<00:00,  1.37it/s]

[enes/random_term] BLEU=100.00 chrF=100.00 term_acc=100.00% macro_cons=1.00 weighted_cons=1.00

Done. C:\Users\vnpnk\Documents\TUM\studies\semester 2\Practical course\terminology-translation\Baseline\openai_translation\metrics_summary.json
